# Notebook 02 - Cleaning & Preprocessing: Vietnam Student Dataset

Notebook này thực hiện quá trình làm sạch và tiền xử lý bộ dữ liệu khảo sát sinh viên Việt Nam nhằm tạo ra bộ dữ liệu sạch, nhất quán và phù hợp cho bước Exploratory Data Analysis (EDA) trong Notebook 03.

Notebook được xây dựng dựa trên cùng logic với Notebook 02 của bộ dữ liệu quốc tế nhằm đảm bảo tính nhất quán trong toàn bộ pipeline nghiên cứu Digital Burnout.


# 0. Set Up
Thiết lập môi trường cần thiết cho quá trình Cleaning & Preprocessing.
Phần này thực hiện:

- Import thư viện.
- Thiết lập cấu hình hiển thị.
- Khai báo đường dẫn input và output.

In [79]:
# Import thư viện xử lý dữ liệu

import warnings

import numpy as np
import pandas as pd

# Import thư viện trực quan hóa

import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập cấu hình hiển thị

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

pd.set_option(
    "display.width",
    None
)

# Ẩn các cảnh báo không cần thiết

warnings.filterwarnings(
    "ignore"
)

print("Đã thiết lập môi trường làm việc.")

Đã thiết lập môi trường làm việc.


# 1. Load Dataset

Đọc dữ liệu khảo sát sinh viên Việt Nam và kiểm tra trạng thái ban đầu trước khi thực hiện cleaning.

Phần này thực hiện:

- Load file CSV.
- Kiểm tra kích thước dataset.
- Kiểm tra kiểu dữ liệu ban đầu.

In [80]:
# Khai báo đường dẫn bộ dữ liệu

dataset_path = "../../data/raw/vietnam_dataset/vn_digital_burnout.csv"

# Đọc bộ dữ liệu khảo sát

survey_dataset = pd.read_csv(
    dataset_path
)

print("Đã tải bộ dữ liệu khảo sát.")

print(f"Số lượng quan sát: {survey_dataset.shape[0]:,}")

print(f"Số lượng biến: {survey_dataset.shape[1]}")

Đã tải bộ dữ liệu khảo sát.
Số lượng quan sát: 697
Số lượng biến: 28


In [81]:
# Kiểm tra thông tin tổng quan của bộ dữ liệu

survey_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 697 entries, 0 to 696
Data columns (total 28 columns):
 #   Column                                                                                                                                           Non-Null Count  Dtype 
---  ------                                                                                                                                           --------------  ----- 
 0   Giới tính của bạn?                                                                                                                               697 non-null    object
 1   Năm sinh?                                                                                                                                        697 non-null    object
 2   Bạn đang ở giai đoạn nào?                                                                                                                        697 non-null    object
 3   Hình thức học tập / làm v

# 2. Data Cleaning

Thực hiện làm sạch dữ liệu khảo sát nhằm đảm bảo tính nhất quán trước khi tiến hành mã hóa và xây dựng biến mục tiêu.

Phần này bao gồm:

- Chuẩn hóa tên biến.
- Kiểm tra và xử lý các câu hỏi Attention Check.
- Chuẩn hóa các giá trị dạng văn bản.

## 2.1 Column Standardization

Chuẩn hóa tên các biến trong bộ dữ liệu khảo sát nhằm thống nhất quy tắc đặt tên và thuận tiện cho quá trình phân tích dữ liệu.

Các tên cột tiếng Việt được chuyển sang tiếng Anh theo chuẩn `snake_case` để đảm bảo tính nhất quán trong toàn bộ pipeline nghiên cứu.

In [82]:
# Kiểm tra danh sách tên cột ban đầu

survey_dataset.columns.tolist()

['Giới tính của bạn?',
 'Năm sinh?',
 'Bạn đang ở giai đoạn nào?',
 'Hình thức học tập / làm việc chính hiện tại:',
 'Mô tả cách bạn dùng thiết bị số hiện tại:',
 'Trung bình mỗi ngày bạn dùng điện thoại và máy tính tổng cộng bao nhiêu giờ  (gộp cả học/làm lẫn giải trí)?',
 'Bạn dành bao nhiêu giờ mỗi ngày cho mạng xã hội (TikTok, Instagram, Facebook, YouTube Shorts...)?\n',
 'Bạn có thường xuyên cuộn feed (TikTok, Reels, Shorts...) liên tục không có mục đích rõ ràng? Nếu có, mỗi ngày khoảng bao lâu?',
 'Bạn có thường dùng điện thoại hoặc máy tính sau 22 giờ đêm?',
 'Bạn nhận khoảng bao nhiêu thông báo trên tất cả ứng dụng mỗi ngày?',
 'Bạn mở khóa điện thoại khoảng bao nhiêu lần mỗi ngày?',
 'Con người cần hô hấp để sống đúng không?',
 'Trong một ngày học/làm việc, bạn chuyển qua lại giữa ứng dụng/tab khoảng bao nhiêu lần?',
 'Bạn tự đánh giá khả năng tập trung của mình dạo này ở mức nào?',
 'Trong 1 giờ học/làm việc, bạn bị kéo ra bởi điện thoại, thông báo hoặc tab khác bao nhiêu lần

In [83]:
# Loại bỏ ký tự xuống dòng và khoảng trắng dư thừa trong tên cột

survey_dataset.columns = (

    survey_dataset.columns
    .str.strip()
    .str.replace("\n", "", regex=False)
    .str.replace(r"\s+", " ", regex=True)

)

In [84]:
# Xây dựng dictionary chuyển đổi tên cột từ tiếng Việt sang tiếng Anh

column_mapping = {

    "Giới tính của bạn?": "gender",
    "Năm sinh?": "birth_year",
    "Bạn đang ở giai đoạn nào?": "education_stage",
    "Hình thức học tập / làm việc chính hiện tại:": "work_mode",
    "Mô tả cách bạn dùng thiết bị số hiện tại:": "device_usage_type",
    "Trung bình mỗi ngày bạn dùng điện thoại và máy tính tổng cộng bao nhiêu giờ (gộp cả học/làm lẫn giải trí)?": "daily_screen_time",
    "Bạn dành bao nhiêu giờ mỗi ngày cho mạng xã hội (TikTok, Instagram, Facebook, YouTube Shorts...)?": "social_media_hours",
    "Bạn có thường xuyên cuộn feed (TikTok, Reels, Shorts...) liên tục không có mục đích rõ ràng? Nếu có, mỗi ngày khoảng bao lâu?": "doomscrolling_duration",
    "Bạn có thường dùng điện thoại hoặc máy tính sau 22 giờ đêm?": "late_night_device_usage",
    "Bạn nhận khoảng bao nhiêu thông báo trên tất cả ứng dụng mỗi ngày?": "notification_count",
    "Bạn mở khóa điện thoại khoảng bao nhiêu lần mỗi ngày?": "smartphone_unlocks",
    "Trong một ngày học/làm việc, bạn chuyển qua lại giữa ứng dụng/tab khoảng bao nhiêu lần?": "app_switch_frequency",
    "Bạn tự đánh giá khả năng tập trung của mình dạo này ở mức nào?": "concentration_score",
    "Trong 1 giờ học/làm việc, bạn bị kéo ra bởi điện thoại, thông báo hoặc tab khác bao nhiêu lần?": "distraction_frequency",
    "Trong một ngày, bạn có bao nhiêu lần ngồi học hoặc làm việc liên tục ít nhất 25 phút mà không mở điện thoại hay chuyển tab?": "focus_sessions",
    "Mỗi ngày bạn dành được bao nhiêu giờ thực sự tập trung sâu vào việc học/làm - không mạng xã hội, không thông báo, không multitask?": "deep_work_hours",
    "Nhìn lại hôm qua, bạn hoàn thành được khoảng bao nhiêu phần trăm những việc mình đã đặt ra (bài tập, deadline, công việc, nhiệm vụ cá nhân...)?": "task_completion_rate",
    "Trung bình bạn ngủ bao nhiêu tiếng mỗi đêm (không tính ngủ trưa)?": "sleep_hours",
    "Dạo này bạn thấy giấc ngủ của mình như thế nào? Khi thức dậy có thấy khỏe và tỉnh táo không?": "sleep_quality",
    "Mức độ có động lực để học tập hoặc làm việc của bạn dạo này ở mức nào?": "motivation_level",
    "Mình cảm thấy kiệt sức chỉ vì phải liên tục nhìn màn hình và xử lý thông tin cả ngày, kể cả khi làm những thứ mình thích.": "digital_exhaustion",
    "Mình thấy căng thẳng, lo lắng khi thấy tin nhắn chưa đọc, thông báo chưa xử lý, hoặc deadline đang chồng chất.": "digital_stress",
    "Tôi cảm thấy mệt mỏi về mặt thể chất sau một ngày làm việc/học tập với các thiết bị số.": "physical_fatigue",
    "Tôi cảm thấy kiệt sức về mặt cảm xúc sau một ngày làm việc/học tập với các thiết bị số.": "emotional_exhaustion",
    "Tôi cảm thấy hiệu suất làm việc/học tập của mình bị suy giảm rõ rệt so với trước đây.": "performance_decline",
    "Tôi cảm thấy mất dần sự hứng thú hoặc hoài nghi về giá trị của công việc/việc học hiện tại.": "loss_of_interest"

}

In [85]:
# Thực hiện chuẩn hóa tên biến

survey_dataset.rename(

    columns=column_mapping,

    inplace=True

)

In [86]:
# Kiểm tra danh sách biến sau khi chuẩn hóa

survey_dataset.columns.tolist()

['gender',
 'birth_year',
 'education_stage',
 'work_mode',
 'device_usage_type',
 'daily_screen_time',
 'social_media_hours',
 'doomscrolling_duration',
 'late_night_device_usage',
 'notification_count',
 'smartphone_unlocks',
 'Con người cần hô hấp để sống đúng không?',
 'app_switch_frequency',
 'concentration_score',
 'distraction_frequency',
 'focus_sessions',
 'deep_work_hours',
 'task_completion_rate',
 'sleep_hours',
 'sleep_quality',
 'motivation_level',
 'digital_exhaustion',
 'digital_stress',
 'physical_fatigue',
 'Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 4)',
 'emotional_exhaustion',
 'performance_decline',
 'loss_of_interest']

## 2.2 Feature Group Definition

Xác định các nhóm biến (feature groups) trong bộ dữ liệu khảo sát sinh viên Việt Nam dựa trên khung lý thuyết Digital Burnout Index (DBI).

Các biến trong khảo sát được phân nhóm theo vai trò trong mô hình nghiên cứu:

- Nhóm biến nhân khẩu học (Demographic Variables): mô tả đặc điểm nền của người tham gia khảo sát.
- Nhóm chỉ báo phơi nhiễm kỹ thuật số (Digital Exposure Indicators): phản ánh mức độ và thói quen sử dụng thiết bị số.
- Nhóm chỉ báo hiệu suất nhận thức (Cognitive Performance Indicators): phản ánh khả năng tập trung, duy trì công việc và hiệu suất học tập.
- Nhóm chỉ báo giấc ngủ và phục hồi (Sleep & Recovery Indicators): phản ánh khả năng nghỉ ngơi và phục hồi năng lượng.
- Nhóm triệu chứng Digital Burnout (Burnout Symptom Indicators): phản ánh trực tiếp các biểu hiện kiệt sức liên quan đến việc sử dụng thiết bị số.

Việc định nghĩa nhóm biến ở bước này giúp đảm bảo tính nhất quán giữa các notebook tiếp theo như EDA, Feature Validation, Modeling và Interpretability.

In [87]:
# Xác định nhóm biến nhân khẩu học

demographic_features = [
    "gender",
    "birth_year",
    "education_stage",
    "work_mode",
    "device_usage_type"
]

In [88]:
# Xác định nhóm chỉ báo phơi nhiễm kỹ thuật số

digital_exposure_features = [
    "daily_screen_time",
    "social_media_hours",
    "doomscrolling_duration",
    "late_night_device_usage",
    "notification_count",
    "smartphone_unlocks",
    "app_switch_frequency"
]

In [89]:
# Xác định nhóm chỉ báo hiệu suất nhận thức

cognitive_features = [
    "concentration_score",
    "distraction_frequency",
    "focus_sessions",
    "deep_work_hours",
    "task_completion_rate",
    "motivation_level"
]

In [90]:
# Xác định nhóm chỉ báo giấc ngủ và phục hồi

sleep_recovery_features = [
    "sleep_hours",
    "sleep_quality"
]

In [91]:
# Xác định nhóm chỉ báo triệu chứng Digital Burnout

burnout_symptom_features = [
    "digital_exhaustion",
    "digital_stress",
    "physical_fatigue",
    "emotional_exhaustion",
    "performance_decline",
    "loss_of_interest"
]

In [92]:
# Xác định nhóm biến kiểm tra chất lượng phản hồi

attention_check_features = [
    "Con người cần hô hấp để sống đúng không?",
    "Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 4)"
]

In [93]:
# Kiểm tra số lượng biến trong từng nhóm

feature_groups_summary = {
    "Demographic": len(demographic_features),
    "Digital Exposure": len(digital_exposure_features),
    "Cognitive Performance": len(cognitive_features),
    "Sleep Recovery": len(sleep_recovery_features),
    "Burnout Symptoms": len(burnout_symptom_features),
    "Attention Check": len(attention_check_features)
}

feature_groups_summary

{'Demographic': 5,
 'Digital Exposure': 7,
 'Cognitive Performance': 6,
 'Sleep Recovery': 2,
 'Burnout Symptoms': 6,
 'Attention Check': 2}

## 2.3 Attention Check Processing

Kiểm tra và loại bỏ các phản hồi khảo sát không hợp lệ dựa trên các câu hỏi Attention Check nhằm đảm bảo chất lượng dữ liệu trước khi thực hiện các bước phân tích và xây dựng mô hình.

Trong quá trình thu thập dữ liệu khảo sát trực tuyến, người tham gia có thể trả lời ngẫu nhiên hoặc không đọc nội dung câu hỏi. Vì vậy, bộ khảo sát được thiết kế với các câu hỏi kiểm tra mức độ chú ý của người trả lời.

Hai biến Attention Check trong dataset bao gồm:

- Câu hỏi kiểm tra kiến thức đơn giản:
    - "Con người cần hô hấp để sống đúng không?"
    - Đáp án hợp lệ: Có

- Câu hỏi kiểm tra sự tiếp tục tham gia khảo sát:
    - "Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 4)"
    - Đáp án hợp lệ: 4

Các phản hồi không vượt qua Attention Check sẽ được loại bỏ khỏi dataset để tránh ảnh hưởng đến kết quả phân tích Digital Burnout.

In [94]:
# Kiểm tra các giá trị xuất hiện trong hai biến Attention Check

for feature in attention_check_features:

    print(f"\n{feature}")

    print(
        survey_dataset[feature]
        .value_counts(dropna=False)
    )


Con người cần hô hấp để sống đúng không?
Con người cần hô hấp để sống đúng không?
Có                 676
Không               14
Mình không biết      7
Name: count, dtype: int64

Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 4)
Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 4)
4    659
3     14
5     10
2      7
1      7
Name: count, dtype: int64


In [95]:
# Xác định điều kiện phản hồi hợp lệ dựa trên Attention Check

attention_check_condition = (

    (survey_dataset["Con người cần hô hấp để sống đúng không?"] == "Có")
    &
    (survey_dataset["Bạn còn đang thực hiện khảo sát này đúng không? (Nếu đúng vui lòng chọn số 4)"] == 4)

)

In [96]:
# Thống kê số lượng phản hồi hợp lệ và không hợp lệ

valid_responses = attention_check_condition.sum()

invalid_responses = len(survey_dataset) - valid_responses


print(f"Số lượng phản hồi hợp lệ: {valid_responses:,}")

print(f"Số lượng phản hồi không hợp lệ: {invalid_responses:,}")

Số lượng phản hồi hợp lệ: 654
Số lượng phản hồi không hợp lệ: 43


In [97]:
# Lọc dataset chỉ giữ lại các phản hồi vượt qua Attention Check

survey_dataset = survey_dataset[
    attention_check_condition
].copy()

In [98]:
# Loại bỏ các biến Attention Check khỏi dataset phân tích

survey_dataset.drop(

    columns=attention_check_features,

    inplace=True

)

In [99]:
# Kiểm tra kích thước dataset sau khi xử lý Attention Check

print(f"Số lượng quan sát sau xử lý: {survey_dataset.shape[0]:,}")

print(f"Số lượng biến sau xử lý: {survey_dataset.shape[1]:,}")

Số lượng quan sát sau xử lý: 654
Số lượng biến sau xử lý: 26


In [100]:
# Kiểm tra các biến còn lại sau khi xử lý Attention Check

survey_dataset.columns.tolist()

['gender',
 'birth_year',
 'education_stage',
 'work_mode',
 'device_usage_type',
 'daily_screen_time',
 'social_media_hours',
 'doomscrolling_duration',
 'late_night_device_usage',
 'notification_count',
 'smartphone_unlocks',
 'app_switch_frequency',
 'concentration_score',
 'distraction_frequency',
 'focus_sessions',
 'deep_work_hours',
 'task_completion_rate',
 'sleep_hours',
 'sleep_quality',
 'motivation_level',
 'digital_exhaustion',
 'digital_stress',
 'physical_fatigue',
 'emotional_exhaustion',
 'performance_decline',
 'loss_of_interest']

## 2.4 Text Value Standardization

Chuẩn hóa các giá trị dạng văn bản trong bộ dữ liệu khảo sát nhằm loại bỏ sự không nhất quán do quá trình nhập liệu từ Google Form và đảm bảo các biến có thể được xử lý chính xác ở các bước mã hóa tiếp theo.

Các biến khảo sát chứa nhiều giá trị dạng text như thời gian sử dụng thiết bị, hình thức học tập, mức độ sử dụng mạng xã hội hoặc các nhóm trả lời theo khoảng.

Trong bước này, dữ liệu được xử lý:

- Loại bỏ khoảng trắng ở đầu và cuối giá trị.
- Loại bỏ khoảng trắng dư thừa giữa các từ.
- Chuẩn hóa ký tự xuống dòng hoặc ký tự đặc biệt.
- Kiểm tra sự nhất quán của các nhóm giá trị.

Lưu ý:
- Bước này chỉ làm sạch dữ liệu text.
- Không thực hiện mã hóa thành số.
- Không thay đổi ý nghĩa của câu trả lời khảo sát.

In [101]:
# Xác định các biến dạng text trong dataset

text_features = (

    survey_dataset
    .select_dtypes(include=["object"])
    .columns
    .tolist()

)


print("Số lượng biến dạng text:", len(text_features))

print(text_features)

Số lượng biến dạng text: 17
['gender', 'birth_year', 'education_stage', 'work_mode', 'device_usage_type', 'daily_screen_time', 'social_media_hours', 'doomscrolling_duration', 'late_night_device_usage', 'notification_count', 'smartphone_unlocks', 'app_switch_frequency', 'distraction_frequency', 'focus_sessions', 'deep_work_hours', 'task_completion_rate', 'sleep_hours']


In [102]:
# Chuẩn hóa giá trị text trong các biến dạng object

for feature in text_features:

    survey_dataset[feature] = (

        survey_dataset[feature]

        .astype(str)

        .str.strip()

        .str.replace("\n", "", regex=False)

        .str.replace(r"\s+", " ", regex=True)

    )

# 3. Survey Response Encoding

Chuyển đổi dữ liệu khảo sát từ dạng câu trả lời ban đầu sang dạng dữ liệu phù hợp cho quá trình phân tích Digital Burnout Index (DBI) và xây dựng mô hình học máy.

Các biến khảo sát được xử lý dựa trên vai trò trong framework Digital Burnout, bao gồm:

- **Demographic Variables:** Giữ nguyên dạng categorical để bảo toàn thông tin mô tả người tham gia.
- **Context Variables:** Giữ nguyên dạng categorical để phản ánh bối cảnh học tập/làm việc và cách sử dụng thiết bị số.
- **Digital Exposure Indicators:** Chuyển đổi các biến hành vi sử dụng thiết bị số sang dạng ordinal theo mức độ phơi nhiễm.
- **Cognitive Performance Indicators:** Chuyển đổi các biến đánh giá khả năng tập trung và hiệu suất nhận thức sang dạng định lượng.
- **Sleep & Recovery Indicators:** Mã hóa các chỉ báo liên quan đến giấc ngủ và khả năng phục hồi.
- **Burnout Symptom Indicators:** Giữ nguyên thang đo Likert để phục vụ tính toán Digital Burnout Score và xây dựng biến mục tiêu.

## 3.1 Demographic Variables Handling

Xác định cách xử lý các biến nhân khẩu học trong bộ dữ liệu khảo sát sinh viên Việt Nam nhằm đảm bảo dữ liệu phù hợp cho cả phân tích mô tả, xây dựng mô hình học máy và triển khai hệ thống đánh giá Digital Burnout.

Nhóm biến nhân khẩu học bao gồm:

- Giới tính.
- Nhóm năm sinh.
- Giai đoạn học tập hoặc làm việc.
- Hình thức học tập / làm việc.
- Loại hình sử dụng thiết bị số.

Các biến này không được chuyển đổi sang dạng số trực tiếp trong bước này.

Lý do:

- Các biến nhân khẩu học không phải là chỉ số đo lường trực tiếp Digital Burnout.
- Giữ dạng categorical giúp bảo toàn ý nghĩa ban đầu.
- Trong giai đoạn Modeling, các biến này sẽ được xử lý bằng Pipeline với OneHotEncoder nếu được đưa vào mô hình.

## 3.1 Demographic Variables

Kiểm tra và chuẩn hóa nhóm biến nhân khẩu học trước khi sử dụng trong quá trình xây dựng mô hình học máy.

Nhóm biến nhân khẩu học bao gồm giới tính, năm sinh và giai đoạn học tập hoặc làm việc. Các biến này được giữ nguyên dưới dạng văn bản vì không có thứ bậc tự nhiên và sẽ được Pipeline tiền xử lý tự động chuyển đổi bằng One-Hot Encoding trong quá trình huấn luyện mô hình.

In [103]:
# Kiểm tra kiểu dữ liệu của nhóm biến nhân khẩu học

survey_dataset[demographic_features].dtypes

gender               object
birth_year           object
education_stage      object
work_mode            object
device_usage_type    object
dtype: object

In [104]:
# Kiểm tra các nhóm giá trị của biến nhân khẩu học

for feature in demographic_features:

    print(f"\n{feature}")

    print(
        survey_dataset[feature]
        .value_counts()
    )


gender
gender
Nữ     415
Nam    239
Name: count, dtype: int64

birth_year
birth_year
2003 - 2005    254
2006 - 2008    174
2000 - 2002    121
2009 - 2012     59
1997 - 1999     46
Name: count, dtype: int64

education_stage
education_stage
Sinh viên đại học / cao đẳng (năm 3-4+)    243
Sinh viên đại học / cao đẳng (năm 1-2)     208
Học sinh THPT                               59
Vừa học vừa đi làm / thực tập               57
Đã đi làm toàn thời gian                    45
Freelancer / tự kinh doanh                  35
Khác                                         7
Name: count, dtype: int64

work_mode
work_mode
Hoàn toàn trực tiếp (đến trường / văn phòng ≥ 4 ngày/tuần)    484
Kết hợp hybrid (2-3 ngày trực tiếp, còn lại online/remote)    122
Hoàn toàn online / remote                                      46
Tự do, không cố định lịch trình                                 2
Name: count, dtype: int64

device_usage_type
device_usage_type
Cân bằng cả học lẫn giải trí                         475


In [105]:
# Kiểm tra giá trị thiếu trong nhóm biến nhân khẩu học

demographic_missing = (
    survey_dataset[demographic_features]
    .isnull()
    .sum()
)


demographic_missing

gender               0
birth_year           0
education_stage      0
work_mode            0
device_usage_type    0
dtype: int64

## 3.2 Context Variables Handling

Xử lý nhóm biến Context Variables nhằm mô tả bối cảnh học tập/làm việc và cách thức sử dụng thiết bị số của người tham gia khảo sát.


Nhóm Context Variables bao gồm:

- Hình thức học tập / làm việc chính (`work_mode`).
- Loại hình sử dụng thiết bị số (`device_usage_type`).

Các biến này không phản ánh trực tiếp mức độ Digital Burnout mà đóng vai trò cung cấp thông tin bối cảnh, giúp giải thích sự khác biệt trong hành vi sử dụng thiết bị số và khả năng thích ứng với môi trường số.

Do đó, nhóm biến này được:

- Giữ nguyên dưới dạng categorical.
- Không chuyển đổi thành điểm số DBI.
- Không áp dụng ordinal encoding.

Các biến này có thể được sử dụng trong:
- Phân tích mô tả và so sánh giữa các nhóm người tham gia.
- Làm biến đầu vào cho mô hình học máy nếu cần thiết.

In [106]:
# Xác định các biến thuộc nhóm Context Variables

context_features = [
    "work_mode",
    "device_usage_type"
]


# Kiểm tra giá trị hiện có của các biến Context Variables

for feature in context_features:

    print(f"\n{feature}")

    print(
        survey_dataset[feature]
        .value_counts()
    )


work_mode
work_mode
Hoàn toàn trực tiếp (đến trường / văn phòng ≥ 4 ngày/tuần)    484
Kết hợp hybrid (2-3 ngày trực tiếp, còn lại online/remote)    122
Hoàn toàn online / remote                                      46
Tự do, không cố định lịch trình                                 2
Name: count, dtype: int64

device_usage_type
device_usage_type
Cân bằng cả học lẫn giải trí                         475
Chủ yếu để giải trí (mạng xã hội, game, xem phim)     95
Chủ yếu để học (LMS, tài liệu, nghiên cứu)            77
Chủ yếu giao tiếp nhóm và làm việc cộng tác            7
Name: count, dtype: int64


In [107]:
# Kiểm tra giá trị thiếu của nhóm Context Variables

context_missing_check = (
    survey_dataset[context_features]
    .isnull()
    .sum()
)


context_missing_check

work_mode            0
device_usage_type    0
dtype: int64

## 3.3 Digital Exposure Encoding

Chuyển đổi các biến thuộc nhóm Digital Exposure từ dữ liệu khảo sát dạng phân loại sang dạng định lượng phù hợp cho quá trình phân tích và xây dựng mô hình học máy.

Nhóm Digital Exposure phản ánh mức độ tiếp xúc và hành vi sử dụng thiết bị số của sinh viên, bao gồm:

- Thời gian sử dụng thiết bị số.
- Thời gian sử dụng mạng xã hội.
- Thời lượng doomscrolling.
- Tần suất sử dụng thiết bị vào ban đêm.
- Số lượng thông báo nhận được.
- Số lần mở khóa điện thoại.
- Tần suất chuyển đổi giữa các ứng dụng.

Các biến này có đặc điểm là dữ liệu dạng ordinal (thứ tự mức độ), do đó được xử lý bằng phương pháp Ordinal Encoding.

Nguyên tắc mã hóa:

- Giá trị nhỏ hơn biểu thị mức độ phơi nhiễm kỹ thuật số thấp hơn.
- Giá trị lớn hơn biểu thị mức độ phơi nhiễm kỹ thuật số cao hơn.
- Không ép buộc tất cả biến về cùng một khoảng điểm.
- Giữ nguyên khoảng giá trị tự nhiên của từng biến dựa trên số lượng mức trả lời trong khảo sát.

Việc chuẩn hóa về cùng thang điểm sẽ được thực hiện trong bước xây dựng DBI Score.

In [108]:
# Xây dựng mapping cho các biến Digital Exposure dựa trên mức độ phơi nhiễm kỹ thuật số

digital_exposure_mapping = {

    "daily_screen_time": {
        "Dưới 4 giờ": 1,
        "4-6 giờ": 2,
        "6-8 giờ": 3,
        "8-10 giờ": 4,
        "Trên 10 giờ": 5
    },

    "social_media_hours": {
        "Dưới 1 giờ": 1,
        "1-2 giờ": 2,
        "2-4 giờ": 3,
        "Trên 4 giờ": 4
    },

    "doomscrolling_duration": {
        "Không bao giờ hoặc rất hiếm": 1,
        "Dưới 30 phút/ngày": 2,
        "30 phút - 1 giờ/ngày": 3,
        "1-2 giờ/ngày": 4,
        "Trên 2 giờ/ngày": 5
    },

    "late_night_device_usage": {
        "Không bao giờ": 1,
        "1-2 lần/tuần": 2,
        "3-5 lần/tuần": 3,
        "Hầu như mỗi tối": 4
    },

    "notification_count": {
        "Dưới 30 thông báo/ngày": 1,
        "30-80 thông báo/ngày": 2,
        "80-150 thông báo/ngày": 3,
        "Trên 150 thông báo/ngày": 4
    },

    "smartphone_unlocks": {
        "Dưới 30 lần/ngày": 1,
        "30-60 lần/ngày": 2,
        "60-100 lần/ngày": 3,
        "Trên 100 lần/ngày": 4
    },

    "app_switch_frequency": {
        "Dưới 20 lần": 1,
        "20-50 lần": 2,
        "50-100 lần": 3,
        "Trên 100 lần": 4
    }

}

In [109]:
# Áp dụng mapping cho nhóm biến Digital Exposure

for feature, mapping in digital_exposure_mapping.items():

    survey_dataset[feature] = survey_dataset[feature].map(mapping)


print("Đã hoàn thành mã hóa nhóm Digital Exposure.")

Đã hoàn thành mã hóa nhóm Digital Exposure.


In [110]:
# Kiểm tra kiểu dữ liệu và giá trị sau khi mã hóa

survey_dataset[digital_exposure_features].info()

<class 'pandas.core.frame.DataFrame'>
Index: 654 entries, 4 to 696
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   daily_screen_time        654 non-null    int64
 1   social_media_hours       654 non-null    int64
 2   doomscrolling_duration   654 non-null    int64
 3   late_night_device_usage  654 non-null    int64
 4   notification_count       654 non-null    int64
 5   smartphone_unlocks       654 non-null    int64
 6   app_switch_frequency     654 non-null    int64
dtypes: int64(7)
memory usage: 40.9 KB


## 3.4 Cognitive Performance Encoding

Chuyển đổi các biến thuộc nhóm Cognitive Performance từ dữ liệu khảo sát sang dạng định lượng phù hợp cho quá trình phân tích Digital Burnout và xây dựng mô hình học máy.

Nhóm Cognitive Performance phản ánh khả năng duy trì hiệu suất nhận thức trong môi trường số, bao gồm:

- Khả năng tập trung.
- Mức độ phân tâm.
- Số lượng phiên học/làm việc tập trung.
- Thời gian làm việc sâu (deep work).
- Tỷ lệ hoàn thành nhiệm vụ.
- Mức độ động lực học tập/làm việc.

Các biến trong nhóm này được chuyển đổi từ dữ liệu khảo sát sang dạng ordinal nhằm biểu diễn mức độ khác nhau của khả năng nhận thức và hiệu suất.

Trong nhóm chỉ báo này tồn tại hai hướng tác động:

- Chỉ báo rủi ro:
    - `distraction_frequency`
    - Giá trị cao hơn thể hiện mức độ phân tâm lớn hơn.

- Chỉ báo bảo vệ:
    - `concentration_score`
    - `focus_sessions`
    - `deep_work_hours`
    - `task_completion_rate`
    - `motivation_level`
    - Giá trị cao hơn thể hiện khả năng duy trì hiệu suất tốt hơn.

Hướng tác động của các biến sẽ được xem xét trong bước xây dựng Digital Burnout Score nhằm đảm bảo tính nhất quán của chỉ số.

In [111]:
# Xác định các biến thuộc nhóm Cognitive Performance

cognitive_features = [
    "concentration_score",
    "distraction_frequency",
    "focus_sessions",
    "deep_work_hours",
    "task_completion_rate",
    "motivation_level"
]


print("Đã xác định nhóm Cognitive Performance.")

Đã xác định nhóm Cognitive Performance.


In [112]:
# Kiểm tra các giá trị hiện có của nhóm Cognitive Performance

for feature in cognitive_features:

    print(f"\n{feature}")

    print(
        survey_dataset[feature]
        .value_counts()
    )


concentration_score
concentration_score
6     167
5     117
7     104
4      68
1      45
8      45
2      33
3      30
10     24
9      21
Name: count, dtype: int64

distraction_frequency
distraction_frequency
6-10 lần                                       257
3-5 lần                                        194
Trên 10 lần (mình gần như bị ngắt liên tục)    145
0-2 lần (mình kiểm soát được khá tốt)           58
Name: count, dtype: int64

focus_sessions
focus_sessions
3-4 lần             283
5 lần trở lên       181
1-2 lần             163
Không có lần nào     27
Name: count, dtype: int64

deep_work_hours
deep_work_hours
2-3 giờ       215
1-2 giờ       190
Trên 3 giờ    135
Dưới 1 giờ    114
Name: count, dtype: int64

task_completion_rate
task_completion_rate
25-50%                                  227
50-75%                                  177
Dưới 25% - hầu như không làm được gì    173
Trên 75% - hoàn thành tốt                77
Name: count, dtype: int64

motivation_level
motivation_

In [113]:
# Mapping các biến Cognitive Performance dạng categorical sang ordinal

cognitive_mapping = {

    "distraction_frequency": {
        "0-2 lần (mình kiểm soát được khá tốt)": 1,
        "3-5 lần": 2,
        "6-10 lần": 3,
        "Trên 10 lần (mình gần như bị ngắt liên tục)": 4
    },

    "focus_sessions": {
        "Không có lần nào": 1,
        "1-2 lần": 2,
        "3-4 lần": 3,
        "5 lần trở lên": 4
    },

    "deep_work_hours": {
        "Dưới 1 giờ": 1,
        "1-2 giờ": 2,
        "2-3 giờ": 3,
        "Trên 3 giờ": 4
    },

    "task_completion_rate": {
        "Dưới 25% - hầu như không làm được gì": 1,
        "25-50%": 2,
        "50-75%": 3,
        "Trên 75% - hoàn thành tốt": 4
    }

}

In [114]:
# Áp dụng ordinal encoding cho các biến categorical

for feature, mapping in cognitive_mapping.items():

    survey_dataset[feature] = (
        survey_dataset[feature]
        .map(mapping)
    )


print("Đã hoàn thành mã hóa nhóm Cognitive Performance.")

Đã hoàn thành mã hóa nhóm Cognitive Performance.


In [115]:
# Kiểm tra dữ liệu sau khi mã hóa

survey_dataset[cognitive_features].head()

,concentration_score,distraction_frequency,focus_sessions,deep_work_hours,task_completion_rate,motivation_level
4,9,4,4,4,2,9
25,8,2,2,2,4,10
37,8,4,1,3,3,3
41,10,2,1,2,2,2
46,3,4,3,2,3,10


## 3.5 Sleep & Recovery Encoding

Chuyển đổi các biến thuộc nhóm Sleep & Recovery sang dạng dữ liệu phù hợp để đánh giá khả năng phục hồi của người tham gia sau quá trình học tập/làm việc và sử dụng thiết bị số.

Nhóm Sleep & Recovery phản ánh trạng thái nghỉ ngơi và khả năng phục hồi, bao gồm:

- `sleep_hours`
- `sleep_quality`

Các biến trong nhóm này được xử lý dựa trên đặc điểm dữ liệu ban đầu:

- `sleep_hours`:
    - Là biến phân loại theo khoảng thời gian ngủ.
    - Được chuyển đổi sang dạng ordinal theo mức độ thời lượng ngủ.

- `sleep_quality`:
    - Là biến đánh giá chất lượng giấc ngủ.
    - Được giữ nguyên hoặc chuyển đổi sang dạng số nếu dữ liệu đã ở dạng Likert.

In [116]:
# Xác định nhóm biến Sleep & Recovery

sleep_features = [
    "sleep_hours",
    "sleep_quality"
]


print("Đã xác định nhóm biến Sleep & Recovery:")
print(sleep_features)

Đã xác định nhóm biến Sleep & Recovery:
['sleep_hours', 'sleep_quality']


In [117]:
# Kiểm tra phân bố giá trị của nhóm Sleep & Recovery

for feature in sleep_features:

    print(f"\n{feature}")

    print(
        survey_dataset[feature]
        .value_counts()
    )


sleep_hours
sleep_hours
5-6 tiếng       221
6-7 tiếng       153
7-8 tiếng       136
Dưới 5 tiếng    107
Trên 8 tiếng     37
Name: count, dtype: int64

sleep_quality
sleep_quality
4     164
6     130
5      88
3      85
7      62
8      47
2      29
9      27
1      12
10     10
Name: count, dtype: int64


In [118]:
# Mapping thời lượng ngủ sang thang ordinal

sleep_hours_mapping = {

    "Dưới 5 tiếng": 1,
    "5-6 tiếng": 2,
    "6-7 tiếng": 3,
    "Trên 8 tiếng": 4,
    "7-8 tiếng": 5

}

In [119]:
# Áp dụng mapping cho biến sleep_hours

survey_dataset["sleep_hours"] = (
    survey_dataset["sleep_hours"]
    .map(sleep_hours_mapping)
)


print("Đã mã hóa biến sleep_hours.")

Đã mã hóa biến sleep_hours.


In [120]:
# Kiểm tra phân bố sau encoding

print("Phân bố sleep_hours sau encoding:")

print(
    survey_dataset["sleep_hours"]
    .value_counts()
    .sort_index()
)

Phân bố sleep_hours sau encoding:
sleep_hours
1    107
2    221
3    153
4     37
5    136
Name: count, dtype: int64


In [121]:
# Kiểm tra sleep_quality sau xử lý

survey_dataset["sleep_quality"].describe()

count    654.00000
mean       5.12844
std        1.92875
min        1.00000
25%        4.00000
50%        5.00000
75%        6.00000
max       10.00000
Name: sleep_quality, dtype: float64

## 3.6 Burnout Symptom Encoding

Xử lý nhóm biến Burnout Symptom nhằm chuẩn hóa các chỉ báo phản ánh trạng thái kiệt sức kỹ thuật số của người tham gia khảo sát.

Nhóm Burnout Symptom bao gồm các biến phản ánh các biểu hiện về thể chất, cảm xúc và hiệu suất liên quan đến Digital Burnout:

- `digital_exhaustion`
- `digital_stress`
- `physical_fatigue`
- `emotional_exhaustion`
- `performance_decline`
- `loss_of_interest`

Các biến này được thu thập dưới dạng thang đo Likert, trong đó giá trị cao hơn thể hiện mức độ xuất hiện triệu chứng Digital Burnout cao hơn.

Do đó:

- Giữ nguyên thang đo ban đầu.
- Không thực hiện ordinal encoding lại.
- Sử dụng trực tiếp cho bước tính toán Digital Burnout Score.

In [122]:
# Xác định nhóm biến Burnout Symptom

burnout_features = [
    "digital_exhaustion",
    "digital_stress",
    "physical_fatigue",
    "emotional_exhaustion",
    "performance_decline",
    "loss_of_interest"
]


print("Đã xác định nhóm biến Burnout Symptom:")
print(burnout_features)

Đã xác định nhóm biến Burnout Symptom:
['digital_exhaustion', 'digital_stress', 'physical_fatigue', 'emotional_exhaustion', 'performance_decline', 'loss_of_interest']


In [123]:
# Kiểm tra giá trị hiện tại của nhóm Burnout Symptom

for feature in burnout_features:

    print(f"\n{feature}")

    print(
        survey_dataset[feature]
        .value_counts()
        .sort_index()
    )


digital_exhaustion
digital_exhaustion
1     24
2     60
3    184
4    256
5    130
Name: count, dtype: int64

digital_stress
digital_stress
1     27
2     45
3     98
4    234
5    250
Name: count, dtype: int64

physical_fatigue
physical_fatigue
1     24
2     74
3    167
4    223
5    166
Name: count, dtype: int64

emotional_exhaustion
emotional_exhaustion
1     30
2     78
3    250
4    174
5    122
Name: count, dtype: int64

performance_decline
performance_decline
1     15
2     76
3    211
4    258
5     94
Name: count, dtype: int64

loss_of_interest
loss_of_interest
1     20
2    187
3    187
4    213
5     47
Name: count, dtype: int64


In [124]:
# Thống kê mô tả nhóm Burnout Symptom

survey_dataset[burnout_features].describe()

,digital_exhaustion,digital_stress,physical_fatigue,emotional_exhaustion,performance_decline,loss_of_interest
count,654.000000,654.000000,654.000000,654.000000,654.000000,654.000000
mean,3.623853,3.970948,3.662080,3.428135,3.519878,3.122324
std,1.018563,1.086213,1.086123,1.064621,0.953146,1.003984
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,3.000000,3.000000,3.000000,3.000000,3.000000,2.000000
50%,4.000000,4.000000,4.000000,3.000000,4.000000,3.000000
75%,4.000000,5.000000,5.000000,4.000000,4.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


# 4. Digital Burnout Index (DBI) Construction


Xây dựng chỉ số Digital Burnout Index (DBI) dựa trên các nhóm chỉ báo đã được xác định trong framework nghiên cứu.

DBI được sử dụng để:

- Định lượng mức độ Digital Burnout của từng người tham gia khảo sát.
- Xây dựng biến mục tiêu phục vụ mô hình học máy.
- Phân loại mức độ Digital Burnout trong các bước phân tích tiếp theo.

Trong nghiên cứu này, Digital Burnout được đánh giá dựa trên nhóm Burnout Symptom Indicators:

- `digital_exhaustion`
- `digital_stress`
- `physical_fatigue`
- `emotional_exhaustion`
- `performance_decline`
- `loss_of_interest`

Các chỉ báo được giữ nguyên thang đo Likert 1–5, trong đó giá trị cao hơn thể hiện mức độ xuất hiện triệu chứng Digital Burnout cao hơn.

DBI Score được tính bằng trung bình cộng của các chỉ báo triệu chứng.

## 4.1 Burnout Symptom Score Calculation

Tính toán điểm triệu chứng Digital Burnout (Burnout Symptom Score) dựa trên các chỉ báo phản ánh trực tiếp trạng thái kiệt sức kỹ thuật số của người tham gia khảo sát.

Burnout Symptom Score được xây dựng dựa trên nhóm Burnout Symptom Indicators:

- `digital_exhaustion`
- `digital_stress`
- `physical_fatigue`
- `emotional_exhaustion`
- `performance_decline`
- `loss_of_interest`

Các biến này đều sử dụng thang đo Likert 1–5, trong đó:

- Giá trị thấp thể hiện mức độ xuất hiện triệu chứng thấp.
- Giá trị cao thể hiện mức độ xuất hiện triệu chứng Digital Burnout cao hơn.

Điểm triệu chứng Digital Burnout được tính bằng giá trị trung bình của 6 chỉ báo Burnout Symptom Indicators:

Burnout Symptom Score =
(digital_exhaustion +
 digital_stress +
 physical_fatigue +
 emotional_exhaustion +
 performance_decline +
 loss_of_interest) / 6

Trong đó:

- Mỗi chỉ báo có giá trị từ 1 đến 5.
- Giá trị càng cao thể hiện mức độ xuất hiện triệu chứng Digital Burnout càng lớn.
- Điểm số cuối cùng nằm trong khoảng từ 1 đến 5.

In [125]:
# Xác định nhóm biến triệu chứng Digital Burnout

burnout_features = [
    "digital_exhaustion",
    "digital_stress",
    "physical_fatigue",
    "emotional_exhaustion",
    "performance_decline",
    "loss_of_interest"
]


print("Đã xác định nhóm Burnout Symptom Indicators:")
print(burnout_features)

Đã xác định nhóm Burnout Symptom Indicators:
['digital_exhaustion', 'digital_stress', 'physical_fatigue', 'emotional_exhaustion', 'performance_decline', 'loss_of_interest']


In [126]:
# Tính điểm trung bình triệu chứng Digital Burnout

survey_dataset["burnout_symptom_score"] = (
    survey_dataset[burnout_features]
    .mean(axis=1)
)


survey_dataset[
    ["burnout_symptom_score"]
].head()

,burnout_symptom_score
4,2.833333
25,3.166667
37,3.000000
41,3.166667
46,3.166667


In [127]:
# Thống kê mô tả Burnout Symptom Score

survey_dataset["burnout_symptom_score"].describe()

count    654.000000
mean       3.554536
std        0.462384
min        1.166667
25%        3.333333
50%        3.666667
75%        3.833333
max        4.500000
Name: burnout_symptom_score, dtype: float64

## 4.2 Digital Burnout Target Score Creation

Xây dựng biến điểm mục tiêu (target score) đại diện cho mức độ Digital Burnout của từng người tham gia khảo sát.

Trong nghiên cứu này, biến mục tiêu được xây dựng dựa trên nhóm Burnout Symptom Indicators, bao gồm:

- `digital_exhaustion`
- `digital_stress`
- `physical_fatigue`
- `emotional_exhaustion`
- `performance_decline`
- `loss_of_interest`

Các chỉ báo này phản ánh trực tiếp các biểu hiện của Digital Burnout và đã được tổng hợp thành biến `burnout_symptom_score`.

Do mục tiêu của mô hình học máy là dự đoán mức độ Digital Burnout dựa trên các yếu tố hành vi sử dụng thiết bị số, khả năng tập trung và khả năng phục hồi, nghiên cứu sử dụng `burnout_symptom_score` làm cơ sở để tạo biến mục tiêu.

In [128]:
# Tạo biến target score

survey_dataset["digital_burnout_score"] = (
    survey_dataset["burnout_symptom_score"]
)


survey_dataset[
    [
        "burnout_symptom_score",
        "digital_burnout_score"
    ]
].head()

,burnout_symptom_score,digital_burnout_score
4,2.833333,2.833333
25,3.166667,3.166667
37,3.000000,3.000000
41,3.166667,3.166667
46,3.166667,3.166667


## 4.3 Burnout Level Definition

Phân loại mức độ Digital Burnout dựa trên giá trị của biến `digital_burnout_score`.

Việc phân loại giúp:

- Trực quan hóa mức độ Digital Burnout trong quá trình phân tích dữ liệu.
- Xác định các nhóm người tham gia có mức độ triệu chứng khác nhau.
- Làm cơ sở xây dựng biến mục tiêu phân loại cho mô hình học máy.

Dựa trên điểm `digital_burnout_score` được tính trên thang đo 1–5, nghiên cứu phân chia mức độ Digital Burnout thành ba nhóm:

| Khoảng điểm | Mức độ |
|---|---|
| < 3.0 | Low |
| 3.0 - < 4.0 | Moderate |
| >= 4.0 | High |

Trong đó:

- **Low**:
    - Mức độ xuất hiện triệu chứng Digital Burnout thấp.

- **Moderate**:
    - Có sự xuất hiện của các triệu chứng Digital Burnout ở mức trung bình.

- **High**:
    - Mức độ xuất hiện triệu chứng Digital Burnout cao.

In [129]:
# Hàm phân loại mức độ Digital Burnout

def classify_burnout_level(score):

    if score < 3:
        return "Low"

    elif score < 4:
        return "Moderate"

    else:
        return "High"

In [130]:
# Tạo biến mức độ Digital Burnout

survey_dataset["dbi_level"] = (
    survey_dataset["digital_burnout_score"]
    .apply(classify_burnout_level)
)


survey_dataset[
    [
        "digital_burnout_score",
        "dbi_level"
    ]
].head()

,digital_burnout_score,dbi_level
4,2.833333,Low
25,3.166667,Moderate
37,3.000000,Moderate
41,3.166667,Moderate
46,3.166667,Moderate


In [131]:
# Kiểm tra số lượng mẫu ở từng mức Digital Burnout

survey_dataset["dbi_level"].value_counts()

# Tính tỷ lệ phần trăm từng nhóm

survey_dataset["dbi_level"].value_counts(
    normalize=True
) * 100

dbi_level
Moderate    68.042813
High        22.477064
Low          9.480122
Name: proportion, dtype: float64

## 4.4 Target Variable Preparation

Chuẩn bị các biến mục tiêu phục vụ cho các bài toán học máy trong các bước tiếp theo.

Nghiên cứu sử dụng hai dạng biến mục tiêu:

1. Regression Target:
- `digital_burnout_score`
- Đại diện cho mức độ Digital Burnout liên tục trên thang điểm 1–5.

2. Classification Target:
- `dbi_level`
- Đại diện cho ba mức độ Digital Burnout:
    - Low
    - Moderate
    - High

Đối với bài toán phân loại, biến `dbi_level` dạng chữ được chuyển đổi sang dạng số để mô hình học máy có thể xử lý.

Quy ước mã hóa:

| DBI Level | Label |
|---|---|
| Low | 0 |
| Moderate | 1 |
| High | 2 |

Biến gốc `dbi_level` được giữ lại nhằm phục vụ việc phân tích và trực quan hóa kết quả.

In [132]:
# Mapping mức độ Digital Burnout sang label số

dbi_mapping = {
    "Low": 0,
    "Moderate": 1,
    "High": 2
}

In [133]:
# Tạo biến target dạng số

survey_dataset["dbi_level_encoded"] = (
    survey_dataset["dbi_level"]
    .map(dbi_mapping)
)


survey_dataset[
    [
        "dbi_level",
        "dbi_level_encoded"
    ]
].head()

,dbi_level,dbi_level_encoded
4,Low,0
25,Moderate,1
37,Moderate,1
41,Moderate,1
46,Moderate,1


In [134]:
# Kiểm tra các giá trị target sau encode

survey_dataset["dbi_level_encoded"].value_counts().sort_index()

dbi_level_encoded
0     62
1    445
2    147
Name: count, dtype: int64

# 5. Export Prepared Dataset

Lưu trữ bộ dữ liệu đã hoàn tất quá trình xử lý để sử dụng cho các giai đoạn tiếp theo của nghiên cứu.

Sau các bước:

- Làm sạch dữ liệu.
- Chuẩn hóa tên biến.
- Xử lý attention check.
- Chuẩn hóa giá trị trả lời khảo sát.
- Mã hóa các nhóm chỉ báo Digital Burnout.
- Xây dựng biến mục tiêu.

Dataset hiện tại đã ở trạng thái phù hợp cho:

- Phân tích khám phá dữ liệu (EDA).
- Kiểm định và lựa chọn đặc trưng.
- Xây dựng mô hình học máy.

## 5.1 Dataset Validation

Kiểm tra lần cuối cấu trúc dataset sau toàn bộ quá trình làm sạch, chuẩn hóa, mã hóa biến và xây dựng target.

Các tiêu chí kiểm tra:

- Số lượng dòng và cột.
- Danh sách biến cuối cùng.
- Kiểu dữ liệu của các biến.
- Giá trị thiếu sau preprocessing.
- Sự tồn tại của các biến mục tiêu.

In [135]:
# Kiểm tra kích thước dataset

survey_dataset.shape

(654, 30)

In [136]:
# Kiểm tra danh sách biến cuối cùng

survey_dataset.columns.tolist()

['gender',
 'birth_year',
 'education_stage',
 'work_mode',
 'device_usage_type',
 'daily_screen_time',
 'social_media_hours',
 'doomscrolling_duration',
 'late_night_device_usage',
 'notification_count',
 'smartphone_unlocks',
 'app_switch_frequency',
 'concentration_score',
 'distraction_frequency',
 'focus_sessions',
 'deep_work_hours',
 'task_completion_rate',
 'sleep_hours',
 'sleep_quality',
 'motivation_level',
 'digital_exhaustion',
 'digital_stress',
 'physical_fatigue',
 'emotional_exhaustion',
 'performance_decline',
 'loss_of_interest',
 'burnout_symptom_score',
 'digital_burnout_score',
 'dbi_level',
 'dbi_level_encoded']

In [137]:
# Kiểm tra missing value

survey_dataset.isnull().sum().sort_values(ascending=False).head(10)

gender                     0
birth_year                 0
education_stage            0
work_mode                  0
device_usage_type          0
daily_screen_time          0
social_media_hours         0
doomscrolling_duration     0
late_night_device_usage    0
notification_count         0
dtype: int64

In [138]:
# Kiểm tra target

survey_dataset[
    [
        "digital_burnout_score",
        "dbi_level",
        "dbi_level_encoded"
    ]
].head()

,digital_burnout_score,dbi_level,dbi_level_encoded
4,2.833333,Low,0
25,3.166667,Moderate,1
37,3.000000,Moderate,1
41,3.166667,Moderate,1
46,3.166667,Moderate,1


## 5.2 Save Processed Dataset

Lưu dataset đã hoàn thành quá trình preprocessing để sử dụng cho các notebook tiếp theo.

Dataset đầu ra bao gồm:

- Biến nhân khẩu học.
- Biến context.
- Các chỉ báo Digital Exposure.
- Các chỉ báo Cognitive Performance.
- Các chỉ báo Sleep & Recovery.
- Các chỉ báo Burnout Symptoms.
- Các biến mục tiêu Digital Burnout.

`vn_digital_burnout_cleaned.csv`

In [140]:
import os

output_dir = "../../data/processed/vietnam_dataset"

os.makedirs(
    output_dir,
    exist_ok=True
)

output_path = "../../data/processed/vietnam_dataset/vn_digital_burnout_cleaned.csv"


survey_dataset.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", output_path)

Saved: ../../data/processed/vietnam_dataset/vn_digital_burnout_cleaned.csv
